In [16]:
import numpy as np
import os
from typing import List, Tuple
from pathlib import Path
from ipynb.fs.full.audio_parser import audio_convert, spectrogram_conversion
from ipynb.fs.full.fingerprint_maker import generate_fingerprints

In [17]:
#need to finish commenting

def create_database(): #highkey redundant if not assigning independent string
    fingerprint_database = dict()
    return fingerprint_database


def add_fingerprints(database: dict, song_id: str, fingerprints: list):
    #adds songs from our library to a dictionary of fingerprints
    for (fm, fn, dt), tm in fingerprints:
        if (fm, fn, dt) not in database:
                database[(fm, fn, dt)] = []
        database[(fm, fn, dt)].append((song_id, tm))

def query_database(database: dict, query_fingerprints: list, freq_tolerance: int = 1, delta_tolerance: int = 1):
    match_counts = {}
    """
    Gets the peaks of an audio file
    Shifts it around each file in the database
    Compares the peaks across each audio file at every time window
    returns the matches
    """
    for (fm, fn, dt), tq in query_fingerprints:
        for df_fm in range(-freq_tolerance, freq_tolerance + 1):
            for df_fn in range(-freq_tolerance, freq_tolerance + 1):
                for d_dt in range(-delta_tolerance, delta_tolerance + 1):
                    fingerprint_key = (fm + df_fm, fn + df_fn, dt + d_dt)

                    if fingerprint_key in database:
                        for song_id, tm in database[fingerprint_key]:
                            time_offset = tm - tq
                            key = (song_id, time_offset)
                            match_counts[key] = match_counts.get(key, 0) + 1

    return match_counts


def get_best_match(match_counts):
    #returns song id and time offset of best match
    #need to add probability feature (if agreed on) using returned highest_count 
    #and remove time offset artifact from highest key. 
    if len(match_counts) == 0:
        return None
    
    highest_key = max(match_counts, key=match_counts.get)
    highest_count = match_counts[highest_key]
    return highest_key, highest_count


In [18]:
database = create_database()

for filename in os.listdir("Music"):
    if not filename.endswith(".wav"):
        continue
    song_id = os.path.splitext(filename)[0]
    song_path = os.path.join("Music", filename)
    samples, sr = audio_convert(song_path)
    _, peaks = spectrogram_conversion(samples, sr)
    fps = generate_fingerprints(peaks, fanout=3)
    add_fingerprints(database, song_id=song_id, fingerprints=fps)

# Query with the recorded clip (same code from earlier)
test_samples, test_sample_rate = audio_convert(os.path.join("Recorded Songs", "black milk clear.wav"))
_, test_peaks = spectrogram_conversion(test_samples, test_sample_rate)
test_fingerprints = generate_fingerprints(test_peaks, fanout=3)

match_counts = query_database(database, test_fingerprints)
best_match = get_best_match(match_counts)
print("Best match:", best_match)

# print("Database fingerprints:", len(my_fingerprints))
print("Test fingerprints:", len(test_fingerprints))
print("Matches:", len(match_counts))

Best match: (('Black Milk', 1778), 2)
Test fingerprints: 63
Matches: 9
